# **Self-attention Pt. II**

**I. Token Sequence**

In [1]:
import torch
import math
import torch.nn as nn

torch.manual_seed(123)
with torch.no_grad():
    '''vocabulary -> 5 tokens in 3 dimensions'''
    #each token is mapped to an index: i.e. (0 -> grass, 1 -> dog, 2 -> eats, 3 -> food, 4 -> a)
    print ("~~~~~~~~~~~~~~~~~~~\nEmbedding of a single token:")
    embedding = nn.Embedding(5,3)

    '''output a single token'''
    # init tensor with index
    wordIdx = torch.tensor((4),dtype=torch.long)
    # take vector representation from embedding layer
    print (embedding(wordIdx))

    '''multiple token output -> sequence'''
    print ("\nEmbedding of a sequence:")
    # init token sequence
    seq = torch.tensor((4,1,2,0),dtype=torch.long)
    # output vector representations
    print (embedding(seq))
    embedSeq = embedding(seq)


~~~~~~~~~~~~~~~~~~~
Embedding of a single token:
tensor([0.2350, 0.6653, 0.3528])

Embedding of a sequence:
tensor([[ 0.2350,  0.6653,  0.3528],
        [-0.2404, -1.1969,  0.2093],
        [-0.9724, -0.7550,  0.3239],
        [-0.1115,  0.1204, -0.3696]])


**II. Init\* Key, Query, Weight matrices**

*In reality, these are learned during transformer training

In [4]:
with torch.no_grad():
    #normally, these are weights that are learnt during training
    q_w = torch.rand(3,3)
    k_w = torch.rand(3,3)
    v_w = torch.rand(3,3)

Based on the embedded input sequence and the learned weights, the queries, keys and values can be computed:

- $Q = seq \times w_q$
- $K = seq \times w_k$
- $V = seq \times w_v$

In [5]:
with torch.no_grad():
    #calculation of query, key, value
    q = torch.matmul(embedSeq,q_w)
    k = torch.matmul(embedSeq,k_w)
    v = torch.matmul(embedSeq,v_w)
    print ("\nQueries:")
    print (q)
    print ("\nKeys:")
    print (k)
    print ("\nValues:")
    print (v)
    print (v.shape)


Queries:
tensor([[ 0.2241,  0.3800,  0.5717],
        [ 0.0531, -0.2545, -0.4865],
        [ 0.0553, -0.5620, -0.9709],
        [-0.1995, -0.1866, -0.2301]])

Keys:
tensor([[ 0.7886,  0.4500,  0.5338],
        [-0.8192, -0.2442, -0.1942],
        [-0.3913, -0.6584, -0.7578],
        [-0.1660, -0.2532, -0.3536]])

Values:
tensor([[ 0.6493,  0.5347,  0.7081],
        [-0.7841, -0.6863, -0.8560],
        [-0.4172, -0.7756, -1.1801],
        [-0.0805, -0.0732, -0.1380]])
torch.Size([4, 3])


**III. Attention weight (or score) calculation**

$scores = Q \times K^T$

Attention scores show the pair-wise influence of each token to all other tokens in the sequences (for a given attention head). The resulting matrix is always: $len_{seq} * len_{seq}$

In [6]:
with torch.no_grad():
    print ("~~~~~~~~~~~~~~~~~~~\nAttention score calculation (Resulting in a matrix of size seqlen * seqlen):")
    attn_scores = torch.matmul(q,k.transpose(0,1))
    print (attn_scores)

~~~~~~~~~~~~~~~~~~~
Attention score calculation (Resulting in a matrix of size seqlen * seqlen):
tensor([[ 0.6529, -0.3874, -0.7711, -0.3356],
        [-0.3324,  0.1132,  0.5155,  0.2276],
        [-0.7275,  0.2805,  1.0841,  0.4764],
        [-0.3641,  0.2537,  0.3753,  0.1617]])


**IV. Full Attention Calculation**

$Attention(Q,K,V)=softmax({QK^T \over \sqrt{d_k}})V$

This results in the contextualized token representations!

In [7]:
with torch.no_grad():
    #Scaled Dot-Product Attention
    attention = torch.matmul(torch.softmax(attn_scores / math.sqrt(k.shape[1]),dim=0),v)
    print ("\nInput sequence (before attention):")
    print (embedding(seq))
    print ("\nOutput sequence (after attention):")
    print (attention)

~~~~~~~~~~~~~~~~~~~
Attention calculation (Resulting in contextualized token representations):

Input sequence (before attention):
tensor([[ 0.2350,  0.6653,  0.3528],
        [-0.2404, -1.1969,  0.2093],
        [-0.9724, -0.7550,  0.3239],
        [-0.1115,  0.1204, -0.3696]])

Output sequence (after attention):
tensor([[ 0.0356, -0.0340, -0.0619],
        [-0.1875, -0.2806, -0.4093],
        [-0.2829, -0.4049, -0.5898],
        [-0.1976, -0.2809, -0.4049]])


# **BERT showcase (word similarity):**

Since all tokens in a sequence influence each other, we are now able to express multiple meanings with one contextualized token!

In [12]:
import transformers
from transformers import BertForMaskedLM
from transformers import BertTokenizer
import torch.nn as nn

def getBertVecsAndTokens(sentence,model,tokenizer):
    inputs = tokenizer(sentence, return_tensors="pt")
    wordVecs = model(**inputs)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze())
    return wordVecs['hidden_states'][-1],tokens

In [14]:
with torch.no_grad():
  tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
  model = BertForMaskedLM.from_pretrained("bert-base-uncased",output_hidden_states=True)

  sentence1 = "The bank of the river"
  sentence2 = "The bank was robbed yesterday"
  sentence3 = "The bank secured the money"

  vecs1,tokens1 = getBertVecsAndTokens(sentence1,model,tokenizer)
  vecs2,tokens2 = getBertVecsAndTokens(sentence2,model,tokenizer)
  vecs3,tokens3 = getBertVecsAndTokens(sentence3,model,tokenizer)

  similarity = nn.CosineSimilarity(dim=2)

  #FOR ALL SENTENCES, BANK WILL BE ON INDEX 2! Notice how even the article 'the' changes!
  print (similarity(vecs1,vecs2)) #bank (river) vs. bank (money)
  print (similarity(vecs1,vecs3)) #bank (river) vs. bank (money)
  print (similarity(vecs2,vecs3)) #bank (money) vs. bank (money)

  print (tokens1)
  print (tokens2)
  print (tokens3)

tensor([[0.8628, 0.5961, 0.5336, 0.4656, 0.3113, 0.4385, 0.9420]])
tensor([[0.8663, 0.6313, 0.5331, 0.3351, 0.5179, 0.4397, 0.9411]])
tensor([[0.9620, 0.8088, 0.8274, 0.4893, 0.4137, 0.7347, 0.9730]])
['[CLS]', 'the', 'bank', 'of', 'the', 'river', '[SEP]']
['[CLS]', 'the', 'bank', 'was', 'robbed', 'yesterday', '[SEP]']
['[CLS]', 'the', 'bank', 'secured', 'the', 'money', '[SEP]']
